In [1]:
import cv2
import numpy as np

In [2]:
src = cv2.imread('apple.png', cv2.IMREAD_GRAYSCALE)
dx = cv2.Sobel(src, -1, 1, 0, delta=120)
dy = cv2.Sobel(src, -1, 0, 1, delta=120)

cv2.imshow('src', src)
cv2.imshow('dx', dx)
cv2.imshow('dy', dy)

cv2.waitKey()
cv2.destroyAllWindows()

## 소벨 마스크 필터를 이용한 에지 검출

In [4]:
src = cv2.imread('beach.jpg', cv2.IMREAD_GRAYSCALE)
dx = cv2.Sobel(src, cv2.CV_32F, 1, 0)
dy = cv2.Sobel(src, cv2.CV_32F, 0, 1)

mag = cv2.magnitude(dx, dy)
mag = np.clip(mag, 0,255).astype(np.uint8)

dst = np.zeros(src.shape[:2], np.uint8)
dst[mag > 120] = 255
cv2.imshow('src', src)
cv2.imshow('mg', mag)
cv2.imshow('dst', dst)

cv2.waitKey()
cv2.destroyAllWindows()

## 캐니 에지

In [5]:
src = cv2.imread('building.png', cv2.IMREAD_GRAYSCALE)

dst = cv2.Canny(src, 50, 150)

cv2.imshow('src', src)
cv2.imshow('dst', dst)

cv2.waitKey()
cv2.destroyAllWindows()

## 허프 변환

In [72]:
src = cv2.imread('building.png', cv2.IMREAD_GRAYSCALE)
edge = cv2.Canny(src, 50, 150)
lines = cv2.HoughLinesP(edge, cv2.COLOR_GRAY2BGR, np.pi/180, 80, minLineLength=100, maxLineGap=10)
dst = cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

if lines is not None:
    for i in range(lines.shape[0]):
        pt1 = lines[i][0][0], lines[i][0][1]
        pt2 = lines[i][0][2], lines[i][0][3]
        cv2.line(dst, pt1, pt2, (0,0,255), 2, cv2.LINE_AA)

cv2.imshow('src', src)
cv2.imshow('dst', dst)

cv2.waitKey()
cv2.destroyAllWindows()

## 원 검출

In [78]:
src = cv2.imread('dial.jpg', cv2.IMREAD_GRAYSCALE)
img = cv2.medianBlur(src, 5)
cimg = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

circles = cv2.HoughCircles(img, cv2.HOUGH_GRADIENT, 1, 50, param1=120, param2=50, minRadius=10, maxRadius=120)
circles = np.uint16(np.around(circles))
for i in circles[0,:]:
    cv2.circle(cimg, (i[0], i[1]), i[2], (0,255,0), 2)

cv2.imshow('src', src)
cv2.imshow('circle', cimg)

cv2.waitKey()
cv2.destroyAllWindows()

# 객체 추적

In [79]:
import cv2
import numpy as np

src = cv2.imread('dog.jpg', cv2.IMREAD_GRAYSCALE)
temp = cv2.imread('dog_template.jpg', cv2.IMREAD_GRAYSCALE)
noise = np.zeros(src.shape, np.int32)
cv2.randn(noise, 50, 10)
res = cv2.matchTemplate(src, temp, cv2.TM_CCOEFF_NORMED)
res_norm = cv2.normalize(res, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
_, maxv, _, maxloc = cv2.minMaxLoc(res)

print('maxv:', maxv)
print('maxloc:', maxloc)

th, tw = temp.shape[:2]
dst = cv2.cvtColor(src, cv2.COLOR_GRAY2BGR)
cv2.rectangle(dst, maxloc, (maxloc[0] + tw, maxloc[1] + th), (0,0,255),2)

cv2.imshow('template', temp)
cv2.imshow('res_norm', res_norm)
cv2.imshow('dst', dst)

cv2.waitKey()
cv2.destroyAllWindows()

maxv: 0.9999838471412659
maxloc: (236, 138)


## 케스케이드 분류기(얼굴)

In [87]:
src = cv2.imread('people.jpg')
c_xml = 'haarcascade_frontalface_alt2.xml'

classifier = cv2.CascadeClassifier(c_xml)

faces = classifier.detectMultiScale(src, minSize=(50,50))
# faces = classifier.detectMultiScale(src)

for (x,y,w,h) in faces:
    cv2.rectangle(src, (x,y,w,h), (255,0,255), 2)

cv2.imshow('src', src)
cv2.waitKey()
cv2.destroyAllWindows()

## HOG 알고리즘과 보행자 검출

In [ ]:
import random, cv2

cap = cv2.VideoCapture('vtest.avi')

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

while True:
    ret, frame = cap.read()
    if not ret:
        break
    detected, _ = hog.detectMultiScale(frame)

    for (x,y,w,h) in detected:
        c = (random.randint(0,255), random.randint(0,255), random.randint(0,255))
        cv2.rectangle(frame, (x,y,w,h), c, 3)
    
    cv2.imshow('frame', frame)
    if cv2.waitKey(10) == 27:
        break

cv2.destroyAllWindows()